# C360 Audience Segmentation (Fresh Demo)
Persona: data scientist; goal: segments + lookalikes entirely in Snowflake.


In [ ]:
import streamlit as st
import altair as alt
import snowflake.snowpark.modin.plugin
from snowflake.snowpark import functions as F
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.window import Window
from snowflake.ml.modeling.preprocessing import StandardScaler
from snowflake.ml.modeling.cluster import KMeans
session = get_active_session()
session.use_database('CROCEVIA_DB'); session.use_schema('GOLD_ANALYTICS')
sales = 'CROCEVIA_DB.BRONZE_DATA.CROCEVIA_SALES_20PCT_STORES'
crm = 'CROCEVIA_DB.RAW_DATA.CROCEVIA_CRM'
products = 'CROCEVIA_DB.BRONZE_DATA.CROCEVIA_PRODUCTS'
st.write('Session ready.')


In [ ]:
# Schema introspection + dynamic selects
sales_df = session.table(sales).select('CUSTOMER_ID','ORDER_ID','SALE_DATE','PRODUCT_ID','QUANTITY','SALES_PRICE_EURO')
crm_tbl = session.table(crm); crm_cols = [f.name for f in crm_tbl.schema.fields]
crm_keep = [c for c in ['CUSTOMER_ID','REGISTRATION_DATE','EMAIL','CITY','MARKETING_OPT_IN'] if c in crm_cols]
crm_df = crm_tbl.select(*crm_keep)
prod_tbl = session.table(products); prod_cols = [f.name for f in prod_tbl.schema.fields]
name_col = 'PRODUCT_NAME' if 'PRODUCT_NAME' in prod_cols else ('PRODUCT' if 'PRODUCT' in prod_cols else None)
prod_keep = ['PRODUCT_ID'] + ([name_col] if name_col else [])
products_df = prod_tbl.select(*prod_keep)
st.write('Sources loaded with dynamic columns.')


In [ ]:
# CRM dedup (latest registration) + join-safety guard
crm_dedup = (crm_df
    .with_column('row_num', F.row_number().over(Window.partition_by('CUSTOMER_ID').order_by(F.col('REGISTRATION_DATE').desc_nulls_last())))
    .filter(F.col('row_num')==1).drop('row_num'))
rfm_base = (sales_df.group_by('CUSTOMER_ID')
    .agg(F.max('SALE_DATE').alias('LAST_PURCHASE_DATE'),
         F.sum('SALES_PRICE_EURO').alias('MONETARY_VALUE'),
         F.count_distinct('ORDER_ID').alias('PURCHASE_FREQUENCY'),
         F.sum('QUANTITY').alias('UNITS'),
         F.count_distinct('PRODUCT_ID').alias('PRODUCT_VARIETY'))
)
rfm_enriched = (rfm_base
    .with_column('RECENCY_DAYS', F.datediff('day', F.col('LAST_PURCHASE_DATE'), F.current_date()))
    .join(crm_dedup, 'CUSTOMER_ID', 'left')
)
total = rfm_enriched.count(); uniq = rfm_enriched.select(F.count_distinct('CUSTOMER_ID')).collect()[0][0]
if total!=uniq:
    st.warning(f'Join produced {total-uniq:,} duplicate rows; enforcing one row per customer.')
    rfm_enriched = rfm_enriched.drop_duplicates(['CUSTOMER_ID'])
st.write('Feature set ready.')


In [ ]:
# Modeling: scale + kmeans + safe writes
feature_cols = ['RECENCY_DAYS','PURCHASE_FREQUENCY','MONETARY_VALUE','UNITS','PRODUCT_VARIETY']
df_for_model = rfm_enriched
for c in feature_cols: df_for_model = df_for_model.with_column(c, F.col(c).cast('DOUBLE'))
scaler = StandardScaler(input_cols=feature_cols, output_cols=[f'SCALED_{c}' for c in feature_cols])
scaled = scaler.fit(df_for_model).transform(df_for_model)
kmeans = KMeans(n_clusters=5, input_cols=[f'SCALED_{c}' for c in feature_cols], random_state=13)
pred = kmeans.fit(scaled).transform(scaled)
if 'SEGMENT_LABEL' not in pred.columns:
    base_pred_col = 'PREDICTION' if 'PREDICTION' in pred.columns else None
    pred = pred.with_column('SEGMENT_LABEL', F.col(base_pred_col)) if base_pred_col else pred
for sc in [f'SCALED_{c}' for c in feature_cols]: pred = pred.with_column(sc, F.coalesce(F.col(sc), F.lit(0.0)))
pred = pred.drop_duplicates(['CUSTOMER_ID'])
pred = pred.with_column('RUN_KEY', F.to_char(F.current_date(), 'YYYYMMDD')).with_column('MODEL_VERSION', F.to_char(F.current_timestamp(), 'YYYYMMDDHH24MISS'))
pred.write.mode('append').save_as_table('CROCEVIA_DB.GOLD_ANALYTICS.C360_CUSTOMER_SEGMENTS_FRESH')
st.success('Segments written to *_FRESH table.')
